# Phase 2: Real-Time Fraud Detection with Continuous Queries

This notebook implements the Continuous Query to detect SMS Pumping / AIT fraud spikes by comparing streaming traffic against historical baselines.

In [ ]:
%%bigquery

WITH continuous_traffic AS (
  SELECT
    CAST(destination AS STRING) AS destination,
    COUNT(*) AS current_count
  FROM
    `fraud-prevention-demo.fraud_data.direct_stream_transactions`
  WHERE
    timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
  GROUP BY
    destination
),
historical_baseline AS (
  SELECT
    CAST(destination AS STRING) AS destination,
    COUNT(*) / (30 * 24 * 60 * 2) AS avg_30s_count
  FROM
    `fraud-prevention-demo.fraud_data.historical_transactions`
  WHERE
    timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  GROUP BY
    destination
)
SELECT
  c.destination,
  c.current_count,
  h.avg_30s_count,
  CURRENT_TIMESTAMP() AS alert_time
FROM
  continuous_traffic c
JOIN
  historical_baseline h ON c.destination = h.destination
WHERE
  c.current_count > (h.avg_30s_count *2)

In [ ]:
import time
from google.cloud import bigquery
from IPython.display import clear_output
import pandas as pd

client = bigquery.Client(project='fraud-prevention-demo')

query = """
WITH continuous_traffic AS (
  SELECT
    LTRIM(CAST(destination AS STRING), '+') AS destination,
    COUNT(*) AS current_count
  FROM
    `fraud-prevention-demo.fraud_data.direct_stream_transactions` -- Ensure correct table name here
  WHERE
    timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
  GROUP BY
    destination
),
historical_baseline AS (
  SELECT
    CAST(destination AS STRING) AS destination,
    COUNT(*) / (30 * 24 * 60 * 2) AS avg_30s_count
  FROM
    `fraud-prevention-demo.fraud_data.historical_transactions`
  WHERE
    timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  GROUP BY
    destination
)
SELECT
  c.destination,
  c.current_count,
  h.avg_30s_count,
  CURRENT_TIMESTAMP() AS alert_time
FROM
  continuous_traffic c
JOIN
  historical_baseline h ON c.destination = h.destination
WHERE
  c.current_count > (h.avg_30s_count * 2)
"""

# Run the query in a loop to simulate continuous updates
for i in range(20):
    df = client.query(query).to_dataframe()
    clear_output(wait=True)
    if not df.empty:
        print("🔥 Fraud Detected!")
        display(df)
    else:
        print("Monitoring stream... No fraud detected yet.")
    time.sleep(5)

## Phase 2.5: Multimodal Fraud Investigation with Gemini

In this section, we demonstrate how to use Gemini in BigQuery to analyze the unstructured data (images and audio) linked to flagged fraudulent transactions.

### Pre-requisites (Run in Terminal)
Ensure you have created a Cloud Resource connection named `gemini-connection` in location `US` and granted it the necessary permissions. See the chat for detailed instructions.

In [ ]:
## Create Object Table for GCS assets

%%bigquery
CREATE OR REPLACE EXTERNAL TABLE `fraud-prevention-demo.fraud_data.fraud_assets`
WITH CONNECTION `fraud-prevention-demo.US.gemini-connection`
OPTIONS (
  object_metadata = 'DIRECTORY',
  uris = ['gs://fraud-prevention-demo-assets/*']
);

In [ ]:
## Create Remote Model pointing to Gemini 2.5 Pro

%%bigquery
CREATE OR REPLACE MODEL `fraud-prevention-demo.fraud_data.gemini_model`
REMOTE WITH CONNECTION `fraud-prevention-demo.US.gemini-connection`
OPTIONS (endpoint = 'gemini-2.5-pro');

In [ ]:
from google.cloud import bigquery
import json

client = bigquery.Client(project='fraud-prevention-demo')

# Pass the destination you want to investigate here
target_destination = "263222222222"

print(f"🔎 Deep diving into destination: {target_destination} using Gemini...")

analysis_query = f"""
WITH flagged_fraud AS (
  SELECT
    destination,
    unstructured_ref
  FROM
    `fraud-prevention-demo.fraud_data.direct_stream_transactions`
  WHERE
    LTRIM(CAST(destination AS STRING), '+') = '{target_destination}'
    AND unstructured_ref IS NOT NULL
  LIMIT 1
)
SELECT
  *
FROM
  ML.GENERATE_TEXT(
    MODEL `fraud-prevention-demo.fraud_data.gemini_model`,
    (
      SELECT
        CONCAT('You are a fraud investigator. Analyze this asset (image or audio) associated with a suspicious transaction to destination ', f.destination, '. Describe what you see/hear and if it looks like phishing or fraud.') AS prompt,
        c.uri,
        f.destination,
        f.unstructured_ref
      FROM
        flagged_fraud f
      JOIN
        `fraud-prevention-demo.fraud_data.fraud_assets` c
      ON
        f.unstructured_ref = c.uri
    ),
    STRUCT(0.2 AS temperature, 1024 AS max_output_tokens)
  )
"""

try:
    df = client.query(analysis_query).to_dataframe()
    if not df.empty:
        print("\n🤖 Gemini Analysis:")
        try:
            result_str = df['ml_generate_text_result'].iloc[0]
            result_json = json.loads(result_str)
            text = result_json['candidates'][0]['content']['parts'][0]['text']
            print(text)
        except Exception as e:
            print(f"Failed to parse Gemini response: {e}")
            print("Raw output:")
            print(df['ml_generate_text_result'].iloc[0])
    else:
        print("\nNo unstructured assets found for this destination to analyze.")
except Exception as e:
    print(f"\nFailed to run Gemini analysis: {e}")